# Data Structure - All DateTime to UTC
---


This notebook filters raw data and flags suspicious values as wrong using different cleaning and quality control techniques


In [ ]:
import os
from pathlib import Path
import requests
import pandas as pd
import numpy as np
import json

import datetime
from datetime import datetime, timedelta
from dateutil import tz


import matplotlib.pyplot as plt
from matplotlib.cm import ScalarMappable
plt.style.use('default')
import matplotlib.colors as colors
import matplotlib.cm as cmx
import matplotlib.dates as mdates
from matplotlib.dates import DateFormatter


In [ ]:

cwd = os.getcwd()
cwd_main = os.path.abspath(os.path.join(cwd, os.pardir))
os.chdir(cwd_main)
print(cwd_main)


import config_tool as cfm


## Define name of city project to work
---


Define the name of the **`city`** (projectname) to upload/save all data during the workflow


In [ ]:

city = os.environ.get("QC_PROJECT_ID", "project_id")


## Import input data of the project - config_project.py 
---


Before running, set `QC_DATA_ROOT` or create a local `path_to_data.txt` from `path_to_data.txt.example`; the submitted archive does not contain data files.


In [ ]:

cwd_project = Path(cfm.cwd_data) / city


os.chdir(cwd_project)
print(cwd_project)


import config_project as cfp


In [ ]:

cwd_data_raw = cfp.cwd_data_raw


cwd_data_raw_wunder = cfp.cwd_data_raw_wunder
cwd_data_raw_netatmo = cfp.cwd_data_raw_netatmo


cwd_data_raw_ows = cfp.cwd_data_raw_ows


cwd_data_str = cfp.cwd_data_str


print(cwd_main)
print(cwd_project)
print(cwd_data_raw)
print(cwd_data_str)
print(cwd_data_raw_wunder)
print(cwd_data_raw_netatmo)
print(cwd_data_raw_ows)


In [ ]:

print("city: ",cfp.city)
print("first date: ",cfp.first_date)
print("last date: ",cfp.last_date)
print("lat,long: ",cfp.lat,cfp.long)
print("plot: ",cfp.plot)

extent = cfp.extent
print("extent: ", extent)

start_date = pd.to_datetime(cfp.first_date, format='%d-%m-%Y %H:%M')
end_date = pd.to_datetime(cfp.last_date, format='%d-%m-%Y %H:%M')
print("data from ", start_date, " to ", end_date)


## Read raw data in projectname\data\10_raw
---


Read data from CWS - Netatmo and Wunderground


In [ ]:

os.chdir(cwd_data_raw_wunder)

name_coordinates_wunder=f"Coordinates_{cfp.city}_CWS_Wunderground.csv"
name_ta_wunder=f"ta_{cfp.city}_{start_date.year}-{end_date.year}_h_CWS_Wunderground.csv"


CWS_coordinates_wunder = pd.read_csv(name_coordinates_wunder)
CWS_ta_wunder = pd.read_csv(name_ta_wunder, index_col='date',parse_dates=True)


CWS_coordinates_wunder.info()
CWS_ta_wunder.info()


In [ ]:

os.chdir(cwd_data_raw_netatmo)

name_coordinates_net=f"Coordinates_{cfp.city}_CWS_Netatmo.csv"
name_ta_net=f"ta_{cfp.city}_{start_date.year}-{end_date.year}_h_CWS_Netatmo.csv"


CWS_coordinates_net = pd.read_csv(name_coordinates_net)
CWS_ta_net = pd.read_csv(name_ta_net, index_col='date',parse_dates=True)


CWS_ta_net.info()
CWS_coordinates_net.info()

print(CWS_ta_net.head())


In [ ]:

print("before :",CWS_ta_net.shape)
print("before :",CWS_ta_wunder.shape)


CWS_ta_net = CWS_ta_net.loc[~CWS_ta_net.index.duplicated(keep='first')]
CWS_ta_net = CWS_ta_net.sort_index()


CWS_ta_wunder = CWS_ta_wunder.loc[~CWS_ta_wunder.index.duplicated(keep='first')]
CWS_ta_wunder = CWS_ta_wunder.sort_index()

print("after :",CWS_ta_net.shape)
print("after :",CWS_ta_wunder.shape)


Read data from OWS (e.g. metoffice) - Omit this step if you don't have data from OWS


In [ ]:


os.chdir(cwd_data_raw_ows)

name_coordinates_ows=f"Coordinates_{cfp.city}_OWS.csv"
name_ta_ows=f"ta_{cfp.city}_{start_date.year}-{end_date.year}_h_OWS.csv"


OWS_coordinates = pd.read_csv(name_coordinates_ows)

OWS_ta = pd.read_csv(name_ta_ows, index_col='date', parse_dates=True)


OWS_ta = OWS_ta.sort_values(by = 'date', ascending =True)


OWS_coordinates.info()
OWS_ta.info()


print(OWS_ta.head())


In [ ]:

print("before :",OWS_ta.shape)


OWS_ta = OWS_ta.loc[~OWS_ta.index.duplicated(keep='first')]
OWS_ta = OWS_ta.sort_index()


print("after :",OWS_ta.shape)


### Filter statations inside the extent


In [ ]:



import pandas as pd

def filter_dataframe_by_extent(df, df_ta, extent, id_name):
    """
    Filters a DataFrame based on the given extent (bounding box).

    Parameters:
        df (pd.DataFrame): Input DataFrame with columns "module_final", "long", "lat".
        extent (tuple): A tuple containing the bounding box (min_long, max_long, min_lat, max_lat).

    Returns:
        pd.DataFrame: Filtered DataFrame containing rows within the extent.
    """
    min_long, max_long, min_lat, max_lat = extent


    filtered_df = df[
        (df['long'] >= min_long) & (df['long'] <= max_long) &
        (df['lat'] >= min_lat) & (df['lat'] <= max_lat)
    ]


    stations_inside_extent = filtered_df[id_name].astype(str).tolist()


    print("Number of stations")
    print("stations inside extent: ", len(stations_inside_extent))
    print("stations_outside_extent: ", len(df) - len(stations_inside_extent))

    print("stations inside extent: ", stations_inside_extent)


    df_ta1 = df_ta.loc[:, df_ta.columns.intersection(stations_inside_extent)]


    return filtered_df, df_ta1


In [ ]:
OWS_coordinates, OWS_ta = filter_dataframe_by_extent(OWS_coordinates, OWS_ta, extent,id_name="src_id")
CWS_coordinates_wunder, CWS_ta_wunder = filter_dataframe_by_extent(CWS_coordinates_wunder, CWS_ta_wunder, extent,id_name="module_final")
CWS_coordinates_net, CWS_ta_net = filter_dataframe_by_extent(CWS_coordinates_net, CWS_ta_net, extent,id_name="module_final")


## Check Data Structure (DateTime) and save in data\11_Structured


In [ ]:
os.chdir(cfm.cwd_scripts_extraction_str)


A projectdata.json file with project variables is created in the script folder


In [ ]:

d = {
     'city':cfp.city,
     'lat':cfp.lat,
     'long':cfp.long,
     'plot':cfp.plot,
     "first_date":cfp.first_date,
     "last_date":cfp.last_date,
     "color_net":cfp.color_net,
     "color_wund":cfp.color_wund,
     "color_ows":cfp.color_ows,
     "color_cws":cfp.color_cws,
    "color_outliers":cfp.color_outliers
     }

with open('projectdata.json', 'w') as fp:
    json.dump(d, fp)


Import functions for data structure


In [ ]:
import structure as str


First, check if data is contaminated


In [ ]:

str.st_datatype(CWS_ta_wunder)


str.st_datatype(CWS_ta_net)


In [ ]:


str.st_datatype(OWS_ta)


Second, visualise DateTime Structure - Check possible structural errors


In [ ]:

with open('projectdata.json', 'r') as fp:
    data = json.load(fp)

city1 = data["city"]
lat = data["lat"]
long = data["long"]


first_date=data["first_date"]
last_date=data["last_date"]


color_net = data["color_net"]
color_wund =data["color_wund"]
color_ows =data["color_ows"]
color_cws = data["color_cws"]
color_outliers = data["color_outliers"]


Verify before time correction


In [ ]:


fig, (ax1, ax2) = plt.subplots(2,1, figsize =(10, 10))
fig.suptitle('VERIFICATION OF STRUCTURAL ERROR - DateTime - BEFORE ',y=0.94)


year1 = datetime.strptime(first_date, '%d-%m-%Y %H:%M').year

initial_date1 = f'2-26-{year1}'
final_date1 = f'2-28-{year1}'


ax1.plot(CWS_ta_net.truncate(before=initial_date1, after=final_date1).iloc[:,0],color=color_net,alpha=0.8,label="CWS - Netatmo",linewidth=1.5)
ax1.plot(CWS_ta_wunder.truncate(before=initial_date1, after=final_date1).iloc[:,1],color=color_wund,alpha=0.8,label="CWS - Wunderground",linewidth=1.5)
ax1.plot(OWS_ta.truncate(before=initial_date1, after=final_date1).iloc[:,0],color=color_ows,alpha=1,label="OWS - Official weather stations",linewidth=1.5)

ax1.plot(CWS_ta_net.truncate(before=initial_date1, after=final_date1),color=color_net,alpha=0.4,linewidth=0.6)
ax1.plot(CWS_ta_wunder.truncate(before=initial_date1, after=final_date1),color=color_wund,alpha=0.6,linewidth=0.6)
ax1.plot(OWS_ta.truncate(before=initial_date1, after=final_date1),color=color_ows,alpha=0.8,linewidth=0.6)


ax1.set(xlabel="Time",
       ylabel="Temperature (ºC)",
       title="Raw CWS data. Winter time")

ax1.set(ylim=(-5, 40))
ax1.legend(loc=1)


date_form = DateFormatter("%y-%m-%d")
ax1.xaxis.set_major_formatter(date_form)


ax1.xaxis.set_major_locator(mdates.DayLocator(interval=1))


initial_date2 = f'07-21-{year1}'
final_date2 = f'07-24-{year1}'

ax2.plot(CWS_ta_net.truncate(before=initial_date2, after=final_date2).iloc[:,0],color=color_net,alpha=0.8,label="CWS - Netatmo",linewidth=1.5)
ax2.plot(CWS_ta_wunder.truncate(before=initial_date2, after=final_date2).iloc[:,0],color=color_wund,alpha=0.8,label="CWS - Wunderground",linewidth=1.5)
ax2.plot(OWS_ta.truncate(before=initial_date2, after=final_date2).iloc[:,0],color=color_ows,alpha=1,label="OWS - Official weather stations",linewidth=1.5)

ax2.plot(CWS_ta_net.truncate(before=initial_date2, after=final_date2),color=color_net,alpha=0.4,linewidth=0.6)
ax2.plot(CWS_ta_wunder.truncate(before=initial_date2, after=final_date2),color=color_wund,alpha=0.6,linewidth=0.6)
ax2.plot(OWS_ta.truncate(before=initial_date2, after=final_date2),color=color_ows,alpha=0.8,linewidth=0.6)


ax2.set(xlabel="Time",
       ylabel="Temperature (ºC)",
       title="Raw CWS data. Summer time")

ax2.set(ylim=(-5, 40))
ax2.legend(loc=1)


date_form = DateFormatter("%y-%m-%d")
ax2.xaxis.set_major_formatter(date_form)


ax2.xaxis.set_major_locator(mdates.DayLocator(interval=1))

os.chdir(cfp.cwd_results_str)
plt.savefig(f"11_DataExtructure_{city}_before.jpg", format='jpg')
plt.show()


Select 1 OWS and a set of CWS in a radius of 2.5km


In [ ]:
from geopy.distance import geodesic


def count_points_within_radius(center_lat, center_long, df, radius_km):
    count = 0
    station_ids = []
    for _, row in df.iterrows():
        point = (row['lat'], row['long'])
        center = (center_lat, center_long)
        if geodesic(center, point).km <= radius_km:
            count += 1
            station_ids.append(row['module_final'])
    return count, station_ids


radius_km = 2.5


net_counts = []
wunder_counts = []
net_station_ids = []
wunder_station_ids = []


for _, ows_row in OWS_coordinates.iterrows():
    center_lat = ows_row['lat']
    center_long = ows_row['long']


    net_count, net_ids = count_points_within_radius(center_lat, center_long, CWS_coordinates_net, radius_km)
    net_counts.append(net_count)
    net_station_ids.append(net_ids)


    wunder_count, wunder_ids = count_points_within_radius(center_lat, center_long, CWS_coordinates_wunder, radius_km)
    wunder_counts.append(wunder_count)
    wunder_station_ids.append(wunder_ids)


results = OWS_coordinates[["src_id","lat","long"]].copy()
results['CWS_net_count'] = net_counts
results['CWS_wunder_count'] = wunder_counts
results['CWS_net_ids'] = net_station_ids
results['CWS_wunder_ids'] = wunder_station_ids

print(results[["src_id","CWS_net_count","CWS_wunder_count"]])


Modify the selected OWS according to the number of CWS in close proximity


In [ ]:


selected_OWS_id = os.environ.get("QC_EXAMPLE_OWS_ID")
if selected_OWS_id is None:
    selected_OWS_id = str(results["src_id"].iloc[0])


selected_row = results[results["src_id"] == (selected_OWS_id)]


List_CWS_ta_net_avg = selected_row["CWS_net_ids"].values[0] if not selected_row.empty else []
List_CWS_ta_wunder_avg = selected_row["CWS_wunder_ids"].values[0] if not selected_row.empty else []

print("List of CWS Netatmo stations:", List_CWS_ta_net_avg)
print("List of CWS Wunderground stations:", List_CWS_ta_wunder_avg)


In [ ]:
if not List_CWS_ta_net_avg or not List_CWS_ta_wunder_avg:
    raise ValueError("No nearby CWS stations were found. Set QC_EXAMPLE_OWS_ID or define station lists from the provided data.")

CWS_ta_wunder_selected = CWS_ta_wunder.loc[:, List_CWS_ta_wunder_avg]
CWS_ta_net_selected = CWS_ta_net.loc[:, List_CWS_ta_net_avg]
OWS_ta_selected = OWS_ta.loc[:, selected_OWS_id]


In [ ]:

start_date1 = '2021-02-25'
end_date1 = '2021-02-28'


CWS_ta_wunder_3days = CWS_ta_wunder_selected.truncate(before=start_date1, after=end_date1)
CWS_ta_net_3days = CWS_ta_net_selected.truncate(before=start_date1, after=end_date1)
OWS_ta_3days = OWS_ta_selected.truncate(before=start_date1, after=end_date1)


plt.figure(figsize=(12, 6))


for col in CWS_ta_wunder_3days.columns:
    plt.plot(CWS_ta_wunder_3days[col], label=f"CWS - Wunderground ({col})", alpha=0.7)


for col in CWS_ta_net_3days.columns:
    plt.plot(CWS_ta_net_3days[col], label=f"CWS - Netatmo ({col})", alpha=0.7)


plt.plot(OWS_ta_3days, label=f"OWS - Official Weather Station ({OWS_ta_3days.name})", alpha=0.9, color='black', linewidth=2)


plt.title("Temperature Data for 3 Days", fontsize=14)
plt.xlabel("Time", fontsize=12)
plt.ylabel("Temperature (ºC)", fontsize=12)


plt.legend()


plt.gca().xaxis.set_major_formatter(DateFormatter("%Y-%m-%d %H:%M"))
plt.gca().xaxis.set_major_locator(mdates.DayLocator(interval=1))


plt.xticks(rotation=45)


plt.grid(True)
plt.show()


In [ ]:
from sklearn.metrics import r2_score


CWS_ta_wunder_avg = CWS_ta_wunder_selected.mean(axis=1)
CWS_ta_net_avg = CWS_ta_net_selected.mean(axis=1)


if isinstance(OWS_ta_selected, pd.DataFrame):
    OWS_ta_avg = OWS_ta_selected.mean(axis=1)
else:
    OWS_ta_avg = OWS_ta_selected.copy()

def safe_r2(obs, ref, label):

    obs, ref = obs.align(ref, join='inner')


    mask = obs.notna() & ref.notna()
    obs = obs[mask]
    ref = ref[mask]

    print(f"{label}: {len(obs)} overlapping valid samples")

    if len(obs) == 0:
        print(f"{label}: no overlapping valid data, R² cannot be computed")
        return None
    if len(obs) < 2:
        print(f"{label}: fewer than 2 samples, R² is not meaningful")
        return None


    return r2_score(obs, ref)

r2_error_wunder = safe_r2(OWS_ta_avg, CWS_ta_wunder_avg, "OWS vs CWS_ta_wunder")
r2_error_net = safe_r2(OWS_ta_avg, CWS_ta_net_avg, "OWS vs CWS_ta_net")

print(f"R² between OWS and CWS_ta_wunder: {r2_error_wunder}")
print(f"R² between OWS and CWS_ta_net: {r2_error_net}")


In [ ]:
print("OWS type:", type(OWS_ta_avg))
print("NET type:", type(CWS_ta_net_avg))

print("OWS length:", len(OWS_ta_avg))
print("NET length:", len(CWS_ta_net_avg))

print("OWS index dtype:", OWS_ta_avg.index.dtype)
print("NET index dtype:", CWS_ta_net_avg.index.dtype)

print("OWS first 5 index values:")
print(OWS_ta_avg.index[:5])

print("NET first 5 index values:")
print(CWS_ta_net_avg.index[:5])

print("OWS time range:", OWS_ta_avg.index.min(), "to", OWS_ta_avg.index.max())
print("NET time range:", CWS_ta_net_avg.index.min(), "to", CWS_ta_net_avg.index.max())

print("OWS non-NaN count:", OWS_ta_avg.notna().sum())
print("NET non-NaN count:", CWS_ta_net_avg.notna().sum())

common_index = OWS_ta_avg.index.intersection(CWS_ta_net_avg.index)
print("Common timestamps:", len(common_index))
print(common_index[:10])


In [ ]:


fig, (ax1, ax2) = plt.subplots(2,1, figsize =(10, 10))
fig.suptitle('VERIFICATION OF STRUCTURAL ERROR - DateTime - BEFORE ',y=0.94)


year1 = datetime.strptime(first_date, '%d-%m-%Y %H:%M').year

initial_date1 = f'2-26-{year1}'
final_date1 = f'2-28-{year1}'


ax1.plot(CWS_ta_net_avg.truncate(before=initial_date1, after=final_date1),color=color_net,alpha=0.8,label="CWS - Netatmo",linewidth=1.5)
ax1.plot(CWS_ta_wunder_avg.truncate(before=initial_date1, after=final_date1),color=color_wund,alpha=0.8,label="CWS - Wunderground",linewidth=1.5)
ax1.plot(OWS_ta_avg.truncate(before=initial_date1, after=final_date1),color=color_ows,alpha=1,label="OWS - Official weather stations",linewidth=1.5)


ax1.set(xlabel="Time",
       ylabel="Temperature (ºC)",
       title="Raw CWS data. Winter time")

ax1.set(ylim=(-5, 40))
ax1.legend(loc=1)


date_form = DateFormatter("%y-%m-%d")
ax1.xaxis.set_major_formatter(date_form)


ax1.xaxis.set_major_locator(mdates.DayLocator(interval=1))


initial_date2 = f'07-21-{year1}'
final_date2 = f'07-24-{year1}'

ax2.plot(CWS_ta_net_avg.truncate(before=initial_date2, after=final_date2),color=color_net,alpha=0.8,label="CWS - Netatmo",linewidth=1.5)
ax2.plot(CWS_ta_wunder_avg.truncate(before=initial_date2, after=final_date2),color=color_wund,alpha=0.8,label="CWS - Wunderground",linewidth=1.5)
ax2.plot(OWS_ta_avg.truncate(before=initial_date2, after=final_date2),color=color_ows,alpha=1,label="OWS - Official weather stations",linewidth=1.5)


ax2.set(xlabel="Time",
       ylabel="Temperature (ºC)",
       title="Raw CWS data. Summer time")

ax2.set(ylim=(-5, 40))
ax2.legend(loc=1)


date_form = DateFormatter("%y-%m-%d")
ax2.xaxis.set_major_formatter(date_form)


ax2.xaxis.set_major_locator(mdates.DayLocator(interval=1))

os.chdir(cfp.cwd_results_str)
plt.savefig(f"11_DataExtructure_{city}_before.jpg", format='jpg')
plt.show()


Code to correct DataTime Structure - all files to UTC


Application to CWS files


First, define the characteristics of the dataset for correction of time structure


In [ ]:


local_timezone = os.environ.get("QC_LOCAL_TIMEZONE", "UTC")


CWS_ta_net_datetime_structure = 'UTC'
CWS_ta_net_delta_direction = "fordward"
CWS_ta_net_delta_time = 30


CWS_ta_wunder_datetime_structure = 'local'
CWS_ta_wunder_delta_direction = False
CWS_ta_wunder_delta_time = 0


In [ ]:

CWS_ta_net1 = str.st_datatime(CWS_ta_net, CWS_ta_net_datetime_structure,  CWS_ta_net_delta_direction, CWS_ta_net_delta_time, local_timezone)


CWS_ta_wunder1 = str.st_datatime(CWS_ta_wunder, CWS_ta_wunder_datetime_structure, CWS_ta_wunder_delta_direction, CWS_ta_wunder_delta_time, local_timezone)

print("Datasets have been processed and corrected to UTC with hourly instant values.")


Check the changes applied to the data - Netatmo:


In [ ]:



time_zone = tz.gettz('UTC')

initial_date1 = pd.Timestamp(f'7-26-{year1}')
final_date1 = pd.Timestamp(f'7-27-{year1}')


initial_date2 = pd.Timestamp(f'{year1}-07-26', tz=time_zone)
final_date2 = pd.Timestamp(f'{year1}-07-27', tz=time_zone)

plt.figure(figsize=(10, 6))
plt.plot(CWS_ta_net.truncate(before=initial_date1, after=final_date1).iloc[:, 0], label="Original Data", color="blue", alpha=0.7)
plt.plot(CWS_ta_net1.truncate(before=initial_date2, after=final_date2).iloc[:, 0], label="Corrected Data", color="red", alpha=0.7)


plt.title("Comparison of Original and Corrected Data", fontsize=14)
plt.xlabel("Time", fontsize=12)
plt.ylabel("Temperature (ºC)", fontsize=12)


plt.legend()


plt.gca().xaxis.set_major_formatter(DateFormatter("%H"))
plt.gca().xaxis.set_major_locator(mdates.HourLocator(interval=1))


plt.grid(True)
plt.show()


Application to OWS (if available)


In [ ]:



local_zone = tz.gettz(os.environ.get("QC_LOCAL_TIMEZONE", "UTC"))
utc_zone = tz.gettz('UTC')

OWS_ta1 = OWS_ta.copy()


OWS_ta1.index = OWS_ta1.index.tz_localize(local_zone, nonexistent='shift_forward', ambiguous=True).tz_convert(utc_zone)


Verify after time correction


In [ ]:


fig, (ax1, ax2) = plt.subplots(2,1, figsize =(10, 10))
fig.suptitle('VERIFICATION OF STRUCTURAL ERROR - dateTime - AFTER',y=0.94)


time_zone = tz.gettz('UTC')


initial_date1 = pd.Timestamp(f'2-26-{year1}').tz_localize(time_zone)
final_date1 = pd.Timestamp(f'2-28-{year1}').tz_localize(time_zone)

ax1.plot(CWS_ta_net1.truncate(before=initial_date1, after=final_date1).iloc[:,0],color=color_net,alpha=0.8,label="CWS - Netatmo",linewidth=1.5)
ax1.plot(CWS_ta_wunder1.truncate(before=initial_date1, after=final_date1).iloc[:,0],color=color_wund,alpha=0.8,label="CWS - Wunderground",linewidth=1.5)
ax1.plot(OWS_ta1.truncate(before=initial_date1, after=final_date1).iloc[:,0],color=color_ows,alpha=1,label="Official weather stations",linewidth=1.5)

ax1.plot(CWS_ta_net1.truncate(before=initial_date1, after=final_date1),color=color_net,alpha=0.4,linewidth=0.6)
ax1.plot(CWS_ta_wunder1.truncate(before=initial_date1, after=final_date1),color=color_wund,alpha=0.6,linewidth=0.6)
ax1.plot(OWS_ta1.truncate(before=initial_date1, after=final_date1),color=color_ows,alpha=0.8,linewidth=0.6)


ax1.set(xlabel="Time",
       ylabel="Temperature (ºC)",
       title="Raw CWS data. Winter time")

ax1.set(ylim=(-5, 40))
ax1.legend(loc=1)


date_form = DateFormatter("%y-%m-%d")
ax1.xaxis.set_major_formatter(date_form)


ax1.xaxis.set_major_locator(mdates.DayLocator(interval=1))


initial_date2 = pd.Timestamp(f'07-21-{year1}').tz_localize(time_zone)
final_date2 = pd.Timestamp(f'07-24-{year1}').tz_localize(time_zone)

ax2.plot(CWS_ta_net1.truncate(before=initial_date2, after=final_date2).iloc[:,0],color=color_net,alpha=0.8,label="CWS - Netatmo",linewidth=1.5)
ax2.plot(CWS_ta_wunder1.truncate(before=initial_date2, after=final_date2).iloc[:,0],color=color_wund,alpha=0.8,label="CWS - Wunderground",linewidth=1.5)
ax2.plot(OWS_ta1.truncate(before=initial_date2, after=final_date2).iloc[:,0],color=color_ows,alpha=1,label="Official weather stations",linewidth=1.5)

ax2.plot(CWS_ta_net1.truncate(before=initial_date2, after=final_date2),color=color_net,alpha=0.4,linewidth=0.6)
ax2.plot(CWS_ta_wunder1.truncate(before=initial_date2, after=final_date2),color=color_wund,alpha=0.6,linewidth=0.6)
ax2.plot(OWS_ta1.truncate(before=initial_date2, after=final_date2),color=color_ows,alpha=0.8,linewidth=0.6)


ax2.set(xlabel="Time",
       ylabel="Temperature (ºC)",
       title="Raw CWS data. Summer time")

ax2.set(ylim=(-5, 40))
ax2.legend(loc=1)


date_form = DateFormatter("%y-%m-%d")
ax2.xaxis.set_major_formatter(date_form)


ax2.xaxis.set_major_locator(mdates.DayLocator(interval=1))

os.chdir(cfp.cwd_results_str)
plt.savefig(f"11_DataExtructure_{city}_after.jpg", format='jpg')
plt.show()


In [ ]:
from sklearn.metrics import r2_score


CWS_ta_wunder1_selected = CWS_ta_wunder1.loc[:,List_CWS_ta_wunder_avg]
CWS_ta_net1_selected = CWS_ta_net1.loc[:,List_CWS_ta_net_avg]
OWS_ta1_selected = OWS_ta1.loc[:,selected_OWS_id]


CWS_ta_wunder1_avg = CWS_ta_wunder1_selected.mean(axis=1)
CWS_ta_net1_avg = CWS_ta_net1_selected.mean(axis=1)
OWS_ta1_avg = OWS_ta1_selected


aligned_ows_wunder1, aligned_wunder1 = OWS_ta1_avg.align(CWS_ta_wunder1_avg, join='inner')


aligned_ows_wunder1 = aligned_ows_wunder1.dropna()
aligned_wunder1 = aligned_wunder1.dropna()


aligned_ows_wunder1, aligned_wunder1 = aligned_ows_wunder1.align(aligned_wunder1, join='inner')


r2_error_wunder1 = r2_score(aligned_ows_wunder1, aligned_wunder1)


aligned_ows_net1, aligned_net1 = OWS_ta1_avg.align(CWS_ta_net1_avg, join='inner')


aligned_ows_net1 = aligned_ows_net1.dropna()
aligned_net1 = aligned_net1.dropna()


aligned_ows_net1, aligned_net1 = aligned_ows_net1.align(aligned_net1, join='inner')


r2_error_net1 = r2_score(aligned_ows_net1, aligned_net1)

print(f"R² error between OWS and CWS_ta_wunder1: {r2_error_wunder1}")
print(f"R² error between OWS and CWS_ta_net1: {r2_error_net1}")


In [ ]:


fig, (ax1, ax2) = plt.subplots(2,1, figsize =(10, 10))
fig.suptitle('VERIFICATION OF STRUCTURAL ERROR - dateTime - AFTER',y=0.94)


time_zone = tz.gettz('UTC')


initial_date1 = pd.Timestamp(f'1-25-{year1}').tz_localize(time_zone)
final_date1 = pd.Timestamp(f'1-29-{year1}').tz_localize(time_zone)

ax1.plot(CWS_ta_net1_avg.truncate(before=initial_date1, after=final_date1),color=color_net,alpha=0.8,label="CWS - Netatmo",linewidth=1.5)
ax1.plot(CWS_ta_wunder1_avg.truncate(before=initial_date1, after=final_date1),color=color_wund,alpha=0.8,label="CWS - Wunderground",linewidth=1.5)
ax1.plot(OWS_ta1_avg.truncate(before=initial_date1, after=final_date1),color=color_ows,alpha=1,label="Official weather stations",linewidth=1.5)


ax1.set(xlabel="Time",
       ylabel="Temperature (ºC)",
       title="Raw CWS data. Winter time")

ax1.set(ylim=(-5, 40))
ax1.legend(loc=1)


date_form = DateFormatter("%y-%m-%d")
ax1.xaxis.set_major_formatter(date_form)


ax1.xaxis.set_major_locator(mdates.DayLocator(interval=1))


initial_date2 = pd.Timestamp(f'06-21-{year1}').tz_localize(time_zone)
final_date2 = pd.Timestamp(f'06-24-{year1}').tz_localize(time_zone)

ax2.plot(CWS_ta_net1_avg.truncate(before=initial_date2, after=final_date2),color=color_net,alpha=0.8,label="CWS - Netatmo",linewidth=1.5)
ax2.plot(CWS_ta_wunder1_avg.truncate(before=initial_date2, after=final_date2),color=color_wund,alpha=0.8,label="CWS - Wunderground",linewidth=1.5)
ax2.plot(OWS_ta1_avg.truncate(before=initial_date2, after=final_date2),color=color_ows,alpha=1,label="Official weather stations",linewidth=1.5)


ax2.set(xlabel="Time",
       ylabel="Temperature (ºC)",
       title="Raw CWS data. Summer time")

ax2.set(ylim=(-5, 40))
ax2.legend(loc=1)


date_form = DateFormatter("%y-%m-%d")
ax2.xaxis.set_major_formatter(date_form)


ax2.xaxis.set_major_locator(mdates.DayLocator(interval=1))

os.chdir(cfp.cwd_results_str)
plt.savefig(f"11_DataExtructure_{city}_after.jpg", format='jpg')
plt.show()


## Save structured data in projectname\data\11_structured


In [ ]:


first_date3 = pd.Timestamp(cfp.first_date).tz_localize(time_zone)
last_date3 = pd.Timestamp(cfp.last_date).tz_localize(time_zone)
print("first date: ",first_date3)
print("last date: ",last_date3)

CWS_ta_net2 = CWS_ta_net1.truncate(before=first_date3, after=last_date3)

CWS_ta_wunder2 = CWS_ta_wunder1.truncate(before=first_date3, after=last_date3)

OWS_ta2 = OWS_ta1.truncate(before=first_date3, after=last_date3)

print(len(CWS_ta_net2))
print(len(CWS_ta_wunder2))
print(len(OWS_ta2))


In [ ]:

os.chdir(cfp.cwd_data_str)


name_coordinates_wunder_str = name_coordinates_wunder.replace(".csv","_str.csv")
name_ta_wunder_str = name_ta_wunder.replace(".csv","_str.csv")

name_coordinates_net_str = name_coordinates_net.replace(".csv","_str.csv")
name_ta_net_str = name_ta_net.replace(".csv","_str.csv")


In [ ]:

CWS_ta_net2.to_csv(name_ta_net_str, index = True , header=True)
CWS_ta_wunder2.to_csv(name_ta_wunder_str, index = True , header=True)

CWS_coordinates_net.to_csv(name_coordinates_net_str, index = False , header=True)
CWS_coordinates_wunder.to_csv(name_coordinates_wunder_str, index = False , header=True)

print("New file has been created: ",name_ta_net_str)
print("New file has been created: ",name_ta_wunder_str)

print("New file has been created: ",name_coordinates_net)
print("New file has been created: ",name_coordinates_wunder)

print("It has been saved in folder: ", cfp.cwd_data_str)
print("List of files in folder")
print(os.listdir())



In [ ]:


name_coordinates_ows=f"Coordinates_{cfp.city}_OWS.csv"
name_coordinates_ows_str = name_coordinates_ows.replace(".csv","_str.csv")

name_ta_ows=f"ta_{cfp.city}_{start_date.year}-{end_date.year}_h_OWS.csv"
name_ta_ows_str = name_ta_ows.replace(".csv","_str.csv")


In [ ]:

OWS_coordinates.to_csv(name_coordinates_ows_str, index = False , header=True)
OWS_ta2.to_csv(name_ta_ows_str, index = True , header=True)

print("New file has been created: ",name_coordinates_ows)
print("New file has been created: ",name_ta_ows)

print("It has been saved in folder: ", cfp.cwd_data_str)
print("List of files in folder")
print(os.listdir())


Eliminate temporal file created in script folder


In [ ]:

os.chdir(cfm.cwd_scripts_extraction_str)

print(os.listdir())


os.remove('projectdata.json')


## Optional - verification of final time structure with raw netatmo data at 30min resolution


In [ ]:
os.chdir(cfp.cwd_data_str)
df1 = pd.read_csv('netatmo_temperature_data_30min2.csv', index_col=0, parse_dates=True)
df2 = pd.read_csv('netatmo_temperature_data_minT2.csv', index_col=0, parse_dates=True)
df3 = pd.read_csv('netatmo_temperature_data_maxT2.csv', index_col=0, parse_dates=True)

df_all = pd.concat([df1, df2, df3], axis=1)

from_zone = tz.gettz('UTC')
df_all.index = df_all.index.tz_localize(from_zone)


In [ ]:



time_zone = tz.gettz('UTC')

initial_date1 = pd.Timestamp(f'7-26-{year1}')
final_date1 = pd.Timestamp(f'7-27-{year1}')


initial_date2 = pd.Timestamp(f'{year1}-07-26', tz=time_zone)
final_date2 = pd.Timestamp(f'{year1}-07-27', tz=time_zone)

plt.figure(figsize=(10, 6))


plt.plot(CWS_ta_net.truncate(before=initial_date1, after=final_date1).iloc[:, 0], label="Netatmo_1h_code_oldversion(project)", color="blue", alpha=0.7)
plt.plot(CWS_ta_net1.truncate(before=initial_date2, after=final_date2).iloc[:, 0], label="Netatmo_1h_code_oldversion(project)_corrected", color="red", alpha=0.7)


plt.plot(df_all.truncate(before=initial_date2, after=final_date2).iloc[:, 0], label="Netatmo_30min_mean", color="orange", alpha=0.7)
plt.plot(df_all.truncate(before=initial_date2, after=final_date2).iloc[:, 1], label="Netatmo_30min_min", color="pink", alpha=0.7)
plt.plot(df_all.truncate(before=initial_date2, after=final_date2).iloc[:, 2], label="Netatmo_30min_max", color="green", alpha=0.7)


plt.title("Comparison of netatmo data extraction methods by resolution and code version", fontsize=14)
plt.xlabel("Time", fontsize=12)
plt.ylabel("Temperature (ºC)", fontsize=12)


plt.legend()


plt.gca().xaxis.set_major_formatter(DateFormatter("%H"))
plt.gca().xaxis.set_major_locator(mdates.HourLocator(interval=1))


os.chdir(cfp.cwd_results_str)
plt.savefig(f"11_DataExtructure_{city}_comparison_netatmo.jpg", format='jpg')


plt.grid(True)
plt.show()
